# Phase 1 - Tokenizer training and corpus tokenization
**Reconstructed** from cells of the original Kaggle notebook (the original file was not recoverable).
Trains a 50k byte-level BPE tokenizer on the Phase-1 legal corpus, then tokenizes all train files into
`uint16` shards (`train_XXXX.npy`) plus `val_tokens.npy` (every 20th document goes to validation).

In [ ]:
# Config
import os
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")   # token stored in Kaggle Secrets (not in code)
OUTPUT_DIR = "/kaggle/working"
REPO_ID = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
VAL_TOKENS = 5_000_000
VOCAB_SIZE = 50_000

In [ ]:
# Download data from HuggingFace
from huggingface_hub import hf_hub_download
train_file_paths = [hf_hub_download(REPO_ID, f, repo_type="dataset", token=HF_TOKEN) for f in TRAIN_FILES]
test_file_path = hf_hub_download(REPO_ID, TEST_FILE, repo_type="dataset", token=HF_TOKEN)
print(train_file_paths)

In [ ]:
!pip install -q ijson

In [ ]:
# Train byte-level BPE tokenizer (50k vocab) on the training text
import ijson, json
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

def text_iterator(limit_docs=None):
    n = 0
    for path in train_file_paths:
        with open(path, "rb") as f:
            for doc in ijson.items(f, "item"):
                text = doc["text"] if isinstance(doc, dict) else doc   # adjust key if your json differs
                yield text
                n += 1
                if limit_docs and n >= limit_docs:
                    return

bpe = ByteLevelBPETokenizer()
bpe.train_from_iterator(text_iterator(), vocab_size=VOCAB_SIZE, min_frequency=2,
                        special_tokens=["<|endoftext|>"])
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=bpe,
    bos_token="<|endoftext|>", eos_token="<|endoftext|>",
    unk_token="<|endoftext|>", pad_token="<|endoftext|>",
)
tok_dir = os.path.join(OUTPUT_DIR, "tokenizer")
tokenizer.save_pretrained(tok_dir)
json.dump({"files": TRAIN_FILES, "vocab_size": VOCAB_SIZE}, open(os.path.join(tok_dir, "trained_on.json"), "w"))
print("vocab size:", len(tokenizer))

In [ ]:
# Sharded tokenization (recovered cell)
import os, json, time, ijson
import numpy as np

SHARD_TOKENS = 100_000_000
BATCH_DOCS   = 500
VAL_EVERY    = 20            # every 20th document -> validation
SHARD_DIR = os.path.join(OUTPUT_DIR, "shards"); os.makedirs(SHARD_DIR, exist_ok=True)
token_dtype = np.uint16 if len(tokenizer) < 65536 else np.uint32
eos = tokenizer.eos_token_id

buf, buf_len, shard_id = [], 0, 0
val_chunks, val_len, doc_count, train_tokens = [], 0, 0, 0

def flush_shard():
    global buf, buf_len, shard_id
    if buf_len == 0: return
    arr = np.concatenate(buf).astype(token_dtype)
    np.save(os.path.join(SHARD_DIR, f"train_{shard_id:04d}.npy"), arr)
    shard_id += 1; buf, buf_len = [], 0

def handle_batch(texts, start_idx):
    global buf_len, val_len, train_tokens
    ids = tokenizer(texts, add_special_tokens=False)["input_ids"]
    for k, toks in enumerate(ids):
        toks = np.array(toks + [eos], dtype=np.int64)
        if (start_idx + k) % VAL_EVERY == 0 and val_len < VAL_TOKENS:
            val_chunks.append(toks); val_len += len(toks)
        else:
            buf.append(toks); buf_len += len(toks); train_tokens += len(toks)
            if buf_len >= SHARD_TOKENS: flush_shard()

t0 = time.time()
for path in train_file_paths:
    batch = []
    with open(path, "rb") as f:
        for doc in ijson.items(f, "item"):
            batch.append(doc["text"] if isinstance(doc, dict) else doc)
            if len(batch) == BATCH_DOCS:
                handle_batch(batch, doc_count); doc_count += len(batch); batch = []
        if batch:
            handle_batch(batch, doc_count); doc_count += len(batch)
    print(path.split("/")[-1], "done", f"{time.time()-t0:.0f}s")
flush_shard()

np.save(os.path.join(OUTPUT_DIR, "val_tokens.npy"), np.concatenate(val_chunks).astype(token_dtype))
json.dump({"train_tokens": int(train_tokens), "val_tokens": int(val_len), "shards": shard_id,
           "dtype": str(np.dtype(token_dtype)), "vocab_size": len(tokenizer)},
          open(os.path.join(OUTPUT_DIR, "tokenized_meta.json"), "w"), indent=2)
print("train tokens:", train_tokens, "val tokens:", val_len, "shards:", shard_id)